# Permutation Importance

**DS4DH Practice Pack · Module 07 — Machine Learning and Interpretability**

*Technique:* Shuffling a feature to measure what the model actually relies on

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/07c_permutation_importance.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The idea is almost too simple: take a fitted model, shuffle one feature's column
so it carries no information, and re-score. However much the score drops is how
much the model was relying on that feature.

It works for any model, and — crucially — it is computed on **held-out** data, so
it measures reliance that generalises rather than reliance the model invented
while memorising.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
TARGET = 'Total'
PREDICTORS = ['renter_owner_gap', 'tot_income', 'log_pop']
RANDOM_STATE = 42

X, y = feat[PREDICTORS], feat[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

gbm = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=3, random_state=RANDOM_STATE).fit(X_train, y_train)
print(f'baseline test R²: {r2_score(y_test, gbm.predict(X_test)):.3f}')

In [ ]:
# Doing it by hand once, so the mechanism is not a black box.
rng = np.random.default_rng(RANDOM_STATE)
baseline = r2_score(y_test, gbm.predict(X_test))

print(f'{"feature":<22}{"R² when shuffled":>18}{"drop":>9}')
print('-' * 49)
for col in PREDICTORS:
    shuffled = X_test.copy()
    shuffled[col] = rng.permutation(shuffled[col].values)
    r2 = r2_score(y_test, gbm.predict(shuffled))
    print(f'{col:<22}{r2:>18.3f}{baseline - r2:>9.3f}')

## Using the library version

One shuffle is one random draw. `permutation_importance` repeats it and reports a
mean and standard deviation, which is what you should quote.

In [ ]:
imp = permutation_importance(gbm, X_test, y_test, n_repeats=30,
                             random_state=RANDOM_STATE, scoring='r2')

order = imp.importances_mean.argsort()[::-1]
print(f'{"feature":<22}{"mean drop":>12}{"sd":>9}')
print('-' * 43)
for i in order:
    print(f'{PREDICTORS[i]:<22}{imp.importances_mean[i]:>12.4f}{imp.importances_std[i]:>9.4f}')

In [ ]:
fig, ax = plt.subplots()
ax.barh([PREDICTORS[i] for i in order[::-1]],
        imp.importances_mean[order[::-1]],
        xerr=imp.importances_std[order[::-1]], color='#3DA5D9')
ax.set_xlabel('drop in test R² when shuffled')
ax.set_title('What the model relies on')
plt.tight_layout()
plt.show()

## The warning that matters

**Importance is not causation.** It says the model's predictions depend on this
column, given the other columns. It does not say changing the thing the column
measures would change housing burden.

Two specific traps:

- **Correlated features share credit.** If two predictors carry the same
  information, shuffling either one alone barely hurts, because the other covers
  for it — so both look unimportant.
- **A feature can be important because of what it proxies for.** `log_pop` is not
  a cause of housing burden; it stands in for urbanisation, market depth, and a
  dozen other things.

In [ ]:
# Demonstrate the correlated-feature trap by duplicating a column.
X2 = X.copy()
X2['income_copy'] = X2['tot_income'] * 1.0

X2tr, X2te, y2tr, y2te = train_test_split(X2, y, test_size=0.25, random_state=RANDOM_STATE)
g2 = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                               random_state=RANDOM_STATE).fit(X2tr, y2tr)
i2 = permutation_importance(g2, X2te, y2te, n_repeats=30,
                            random_state=RANDOM_STATE, scoring='r2')

print(f'{"feature":<22}{"mean drop":>12}')
print('-' * 34)
for name, val in sorted(zip(X2.columns, i2.importances_mean),
                        key=lambda t: -t[1]):
    print(f'{name:<22}{val:>12.4f}')
print()
print('Income now looks far less important than it did — its twin covers for it.')
print('Nothing about the world changed. Only the feature set did.')

In [ ]:
print(f'{"":<24}{"alone":>10}{"with a twin":>14}')
print('-' * 48)
solo = dict(zip(PREDICTORS, imp.importances_mean))
twin = dict(zip(X2.columns, i2.importances_mean))
print(f'{"tot_income":<24}{solo["tot_income"]:>10.4f}{twin["tot_income"]:>14.4f}')
print()
print('Always check correlations before reading an importance ranking:')
print(feat[PREDICTORS].corr().round(2).to_string())

### 🔧 Your turn 1

Add `Renter` to `PREDICTORS` and re-run the importance ranking.

`Total` STIR is partly composed of renter STIR, so this is close to leaking the
answer. What happens to the other features' importances, and why is this a
cautionary tale about feature selection rather than about the method?

### 🔧 Your turn 2

Change `scoring='r2'` to `scoring='neg_mean_absolute_error'`.

Does the ranking change? Importance is always importance *for a particular
metric* — the ranking is not a property of the data alone.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** `Renter` dominates and everything else collapses toward zero.
That is target leakage: `Total` is a household-weighted blend of owner and renter
burden, so you have handed the model part of its own answer. Test R² will look
excellent and the model will be useless for anything you would actually want it
for. Permutation importance did its job correctly — it reported that the model
relies on the leaked column. The failure was in choosing the feature set, which
no interpretability method can rescue you from.

**Your turn 2.** The ranking usually stays similar here but the magnitudes change,
and with a different dataset the order can flip. R² penalises large errors
quadratically, MAE linearly — so a feature that mainly helps with a few extreme
CSDs looks more important under R² than under MAE. Quote the metric alongside the
importance, always.

</details>

## Where this stops

Module 07's honest summary: a boosted model predicts municipal housing burden
better than a linear one, and income and market size are what it leans on. None
of that licenses a claim that raising incomes would lower burden by any particular
amount.

Next: Module 08 stops modelling an existing number and builds a new one.